In [6]:
import torch
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.nn import global_mean_pool


class GAT(torch.nn.Module):
    def __init__(self, hidden_channels):
        super(GAT, self).__init__()
        torch.manual_seed(12345)
        self.conv1 = GATConv(320, hidden_channels)
        self.conv2 = GATConv(hidden_channels, hidden_channels)
        self.conv3 = GATConv(hidden_channels, hidden_channels)
        self.lin = Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        # 1. Obtain node embeddings 
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = x.relu()
        x = self.conv3(x, edge_index)

        # 2. Readout layer
        x = global_mean_pool(x, batch)  # [batch_size, hidden_channels]

        # 3. Apply a final classifier
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.lin(x)
        
        return x

In [7]:
from torch_geometric.data import Dataset
import torch
import os

class PatientGraphDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        super().__init__()
        self.root_dir = root_dir
        self.files = [os.path.join(root_dir, f) for f in os.listdir(root_dir) if f.endswith('.pt')]
        self.transform = transform

    def len(self):
        return len(self.files)

    def get(self, idx):
        self.graph, self.df = torch.load(self.files[idx], weights_only=False)
        if self.transform:
            self.graph = self.transform(self.graph)
        return self.graph


In [8]:
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

downsample_size = 5000
full_dataset = PatientGraphDataset(f"/dsi/sbm/chen2/graph_data/{downsample_size}/0.05_True_True/")

# Extract all labels first
all_labels = []
for i in range(len(full_dataset)):
    graph = full_dataset[i]
    all_labels.append(graph.y.item())

# Create indices array
indices = np.arange(len(full_dataset))

# First split: train (70%) vs test (30%)
train_indices, test_indices = train_test_split(
    indices, test_size=0.3, stratify=all_labels, random_state=42
)

# # Second split: test (20%) vs eval (10%) from the temp (30%)
# temp_labels = [all_labels[i] for i in temp_indices]
# test_indices, eval_indices = train_test_split(
#     temp_indices, test_size=0.33, stratify=temp_labels, random_state=42  # 0.33 of 30% = 10%
# )

# Create stratified datasets
from torch.utils.data import Subset

train_dataset_stratified = Subset(full_dataset, train_indices)
test_dataset_stratified = Subset(full_dataset, test_indices)
# eval_dataset_stratified = Subset(full_dataset, eval_indices)

# Create new data loaders
train_loader_stratified = DataLoader(train_dataset_stratified, batch_size=2, shuffle=True)
test_loader_stratified = DataLoader(test_dataset_stratified, batch_size=2)
# eval_loader_stratified = DataLoader(eval_dataset_stratified, batch_size=4)

In [9]:
import torch

# Check if CUDA is available and set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = GAT(hidden_channels=16).to(device)  # Move model to device
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

def train():
    model.train()
    total_loss = 0
    num_batches = 0

    for data in train_loader_stratified:  # Use stratified loader
        optimizer.zero_grad()  # Clear gradients BEFORE forward pass
        data = data.to(device)  # Move data to device
        out = model(data.x, data.edge_index, data.batch)  # Perform a single forward pass.
        loss = criterion(out, data.y)  # Compute the loss.
        loss.backward()  # Derive gradients.
        optimizer.step()  # Update parameters based on gradients.
         
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches  # Return average loss

def test(loader):
     model.eval()
     
     correct = 0
     with torch.no_grad():  # Disable gradient computation for evaluation
        for data in loader:  # Iterate in batches over the training/test dataset.
            data = data.to(device)  # Move data to device
            out = model(data.x, data.edge_index, data.batch)  
            pred = out.argmax(dim=1)  # Use the class with highest probability.
            correct += int((pred == data.y).sum())  # Check against ground-truth labels.
     return correct / len(loader.dataset)  # Derive ratio of correct predictions.


for epoch in range(1, 20):
    avg_loss = train()
    train_acc = test(train_loader_stratified)
    test_acc = test(test_loader_stratified)
    print(f'Epoch: {epoch:03d}, Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')

Using device: cuda


RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 12778 but got size 12322 for tensor number 1 in the list.

In [ ]:
full_dataset[0]

Data(x=[500, 320], edge_index=[2, 501], edge_attr=[501], y=[1], node_weights=[500])